In [0]:
%pip install shap

In [0]:
%pip install xgboost

In [0]:
# Importing Libraries

import numpy as np
import pandas as pd
import shap
import mlflow
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, udf, struct
from pyspark.sql.types import ArrayType, StringType, DoubleType, StructType, StructField
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/explainability")

# Feature names

APP_FEATURE_NAMES = [
    "log_income",
    "address_stability",
    "under_25",
    "name_email_similarity",
    "days_since_request",
    "zip_count_4w"
]

TXN_FEATURE_NAMES = [
    "log_amount",
    "amount_balance_ratio",
    "transaction_hour",
    "transaction_dayofweek",
    "merchant_fraud_rate",
    "customer_avg_transaction_amount",
    "merchant_category_index",
    "device_type_index",
    "channel_grouped_index",
    "location_city_grouped_index"
]

# Loading Models and data

def load_data_as_numpy(train_table, test_table, sample_n=50000):
    """Load Spark tables and convert to numpy with stratified sampling."""
    train_df = spark.table(train_table).withColumn(
        "is_fraud", col("is_fraud").cast("double")
    )
    test_df = spark.table(test_table).withColumn(
        "is_fraud", col("is_fraud").cast("double")
    )

    # Stratified sample for SHAP
    fraud_sample = test_df.filter("is_fraud = 1").limit(sample_n // 10)
    legit_sample = test_df.filter("is_fraud = 0").limit(sample_n)
    test_sample  = fraud_sample.union(legit_sample)

    def extract(df):
        arr = df.withColumn("features_arr", vector_to_array("features"))
        pdf = arr.select("features_arr", "is_fraud").toPandas()
        X   = np.array(pdf["features_arr"].tolist())
        y   = pdf["is_fraud"].values
        return X, y

    X_train, y_train = extract(train_df)
    X_test,  y_test  = extract(test_sample)

    # Normalization
    scaler   = StandardScaler()
    scaler.fit(X_train[y_train == 0])
    X_train  = scaler.transform(X_train)
    X_test   = scaler.transform(X_test)

    return X_train, y_train, X_test, y_test, scaler


print("Loading application data...")
X_app_train, y_app_train, X_app_test, y_app_test, scaler_app = load_data_as_numpy(
    "workspace.ml_layer.application_train_features",
    "workspace.ml_layer.application_test_features"
)

print("Loading transaction data...")
X_txn_train, y_txn_train, X_txn_test, y_txn_test, scaler_txn = load_data_as_numpy(
    "workspace.ml_layer.transaction_train_features",
    "workspace.ml_layer.transaction_test_features",
    sample_n=100000
)

# Reloading XGboost models

import xgboost as xgb

def train_xgb(X_train, y_train, X_test, y_test, dataset_name):
    fraud_count      = int(y_train.sum())
    legit_count      = len(y_train) - fraud_count
    scale_pos_weight = legit_count / fraud_count

    model = xgb.XGBClassifier(
        n_estimators     = 300,
        max_depth        = 6,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        scale_pos_weight = scale_pos_weight,
        eval_metric      = "aucpr",
        random_state     = 42,
        early_stopping_rounds=30,
        n_jobs           = -1
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=50
    )
    return model

print("Training XGBoost for applications...")
xgb_app = train_xgb(X_app_train, y_app_train, X_app_test, y_app_test, "applications")

print("Training XGBoost for transactions...")
xgb_txn = train_xgb(X_txn_train, y_txn_train, X_txn_test, y_txn_test, "transactions")

# SHAP interpretation

def run_shap_analysis(model, X_test, feature_names, dataset_name, n_samples=2000):
    """
    Compute SHAP values and generate summary + dependence plots.
    n_samples: subset for visualization — full SHAP on 50K rows is slow
    """
    print(f"\n  Computing SHAP values for {dataset_name}...")

    # Useing a background sample for the explainer
    background = shap.sample(X_test, 100, random_state=42)
    explainer  = shap.TreeExplainer(model, background)

    # Computing on sample for speed
    idx          = np.random.choice(len(X_test), min(n_samples, len(X_test)), replace=False)
    X_sample     = X_test[idx]
    shap_values  = explainer.shap_values(X_sample)

    # For binary classification XGBoost returns single array
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    # Plotting SHAP Summary
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values,
        X_sample,
        feature_names=feature_names,
        show=False,
        max_display=len(feature_names)
    )
    plt.title(f"SHAP Feature Impact — {dataset_name}", fontsize=14, pad=20)
    plt.tight_layout()
    summary_path = f"/tmp/shap_summary_{dataset_name}.png"
    plt.savefig(summary_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {summary_path}")

    # Plotting SHAP Bar
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values,
        X_sample,
        feature_names=feature_names,
        plot_type="bar",
        show=False,
        max_display=len(feature_names)
    )
    plt.title(f"SHAP Mean |Value| — {dataset_name}", fontsize=14, pad=20)
    plt.tight_layout()
    bar_path = f"/tmp/shap_bar_{dataset_name}.png"
    plt.savefig(bar_path, dpi=150, bbox_inches="tight")
    plt.show()

    # Plotting Dependence plot for top feature 
    top_feature_idx = np.abs(shap_values).mean(axis=0).argmax()
    top_feature     = feature_names[top_feature_idx]

    plt.figure(figsize=(10, 6))
    shap.dependence_plot(
        top_feature_idx,
        shap_values,
        X_sample,
        feature_names=feature_names,
        show=False
    )
    plt.title(f"SHAP Dependence — {top_feature} | {dataset_name}", fontsize=14)
    plt.tight_layout()
    dep_path = f"/tmp/shap_dependence_{dataset_name}.png"
    plt.savefig(dep_path, dpi=150, bbox_inches="tight")
    plt.show()

    # Returning mean SHAP importance table
    mean_shap = pd.DataFrame({
        "feature"         : feature_names,
        "mean_abs_shap"   : np.abs(shap_values).mean(axis=0),
        "mean_shap"       : shap_values.mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    mean_shap["rank"] = range(1, len(mean_shap) + 1)

    print(f"\n  SHAP Feature Ranking — {dataset_name}")
    print(mean_shap.to_string(index=False))

    return shap_values, mean_shap, explainer, idx


with mlflow.start_run(run_name="SHAP_applications"):
    shap_vals_app, shap_importance_app, explainer_app, idx_app = run_shap_analysis(
        xgb_app, X_app_test, APP_FEATURE_NAMES, "applications"
    )
    mlflow.log_dict(
        shap_importance_app.to_dict(orient="records"),
        "shap_importance_applications.json"
    )

with mlflow.start_run(run_name="SHAP_transactions"):
    shap_vals_txn, shap_importance_txn, explainer_txn, idx_txn = run_shap_analysis(
        xgb_txn, X_txn_test, TXN_FEATURE_NAMES, "transactions"
    )
    mlflow.log_dict(
        shap_importance_txn.to_dict(orient="records"),
        "shap_importance_transactions.json"
    )


# Feature Importance

def plot_xgb_feature_importance(model, feature_names, dataset_name):
    importance_types = ["gain", "weight", "cover"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, imp_type in zip(axes, importance_types):
        scores = model.get_booster().get_score(importance_type=imp_type)
        # Map f0, f1... back to real feature names
        named_scores = {
            feature_names[int(k.replace("f", ""))]: v
            for k, v in scores.items()
            if k.replace("f", "").isdigit()
            and int(k.replace("f", "")) < len(feature_names)
        }
        sorted_scores = dict(sorted(named_scores.items(),
                                    key=lambda x: x[1], reverse=True))

        ax.barh(list(sorted_scores.keys()), list(sorted_scores.values()),
                color="#2563EB", alpha=0.8)
        ax.set_title(f"Importance by {imp_type.upper()}", fontsize=12)
        ax.set_xlabel(imp_type)
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.3)

    fig.suptitle(f"XGBoost Feature Importance — {dataset_name}",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    path = f"/tmp/xgb_importance_{dataset_name}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return named_scores

importance_app = plot_xgb_feature_importance(xgb_app, APP_FEATURE_NAMES, "applications")
importance_txn = plot_xgb_feature_importance(xgb_txn, TXN_FEATURE_NAMES, "transactions")


# Reason Codes

def generate_reason_codes(
    model, explainer, X, feature_names,
    dataset_name, n_examples=10
):
    """
    Generate human-readable reason codes for individual predictions.
    Returns a DataFrame with one row per transaction/application,
    showing the top features driving the fraud score.
    """

    # Computing SHAP for these specific examples
    shap_vals = explainer.shap_values(X[:n_examples])
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]

    y_proba = model.predict_proba(X[:n_examples])[:, 1]

    records = []
    for i in range(n_examples):
        fraud_score = y_proba[i]
        sv          = shap_vals[i]

        # Sorting features by absolute SHAP contribution
        feature_impacts = sorted(
            zip(feature_names, sv, X[i]),
            key=lambda x: abs(x[1]),
            reverse=True
        )

        # Top 3 reason codes
        reasons = []
        for feat_name, shap_val, feat_val in feature_impacts[:3]:
            direction = "↑ increases" if shap_val > 0 else "↓ decreases"
            reasons.append(
                f"{feat_name} ({feat_val:.3f}) {direction} fraud risk "
                f"[SHAP: {shap_val:+.4f}]"
            )

        # Risk tier — maps score to business-readable category
        if fraud_score >= 0.80:
            risk_tier = "🔴 HIGH RISK"
        elif fraud_score >= 0.40:
            risk_tier = "🟡 MEDIUM RISK"
        else:
            risk_tier = "🟢 LOW RISK"

        records.append({
            "example_id"  : i,
            "fraud_score" : round(fraud_score, 4),
            "risk_tier"   : risk_tier,
            "reason_1"    : reasons[0] if len(reasons) > 0 else "",
            "reason_2"    : reasons[1] if len(reasons) > 1 else "",
            "reason_3"    : reasons[2] if len(reasons) > 2 else "",
        })

    reason_df = pd.DataFrame(records)
    return reason_df


print("\n" + "="*70)
print("  REASON CODES — Applications (sample of 10)")
print("="*70)
reason_codes_app = generate_reason_codes(
    xgb_app, explainer_app,
    X_app_test, APP_FEATURE_NAMES,
    "applications", n_examples=10
)
display(reason_codes_app)

print("\n" + "="*70)
print("  REASON CODES — Transactions (sample of 10)")
print("="*70)
reason_codes_txn = generate_reason_codes(
    xgb_txn, explainer_txn,
    X_txn_test, TXN_FEATURE_NAMES,
    "transactions", n_examples=10
)
display(reason_codes_txn)

spark.createDataFrame(reason_codes_app).write.format("delta") \
    .mode("overwrite").saveAsTable("workspace.ml_layer.reason_codes_applications")

spark.createDataFrame(reason_codes_txn).write.format("delta") \
    .mode("overwrite").saveAsTable("workspace.ml_layer.reason_codes_transactions")

print("\nReason codes saved to Delta tables")

# Individual Prediction Waterfall

def explain_single_prediction(
    model, explainer, X, feature_names,
    dataset_name, example_idx=0
):
    """Show waterfall plot for one specific prediction — ideal for defense demo."""
    x_single    = X[example_idx:example_idx+1]
    shap_vals   = explainer.shap_values(x_single)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]

    fraud_score = model.predict_proba(x_single)[0, 1]
    print(f"\n  Explaining prediction {example_idx} | "
          f"Fraud score: {fraud_score:.4f} | "
          f"{'🔴 FRAUD' if fraud_score > 0.5 else '🟢 LEGIT'}")

    shap.waterfall_plot(
        shap.Explanation(
            values        = shap_vals[0],
            base_values   = explainer.expected_value
                            if not isinstance(explainer.expected_value, list)
                            else explainer.expected_value[1],
            data          = x_single[0],
            feature_names = feature_names
        ),
        show=False
    )
    plt.title(f"Individual Prediction Explanation — {dataset_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"/tmp/waterfall_{dataset_name}_{example_idx}.png",
                dpi=150, bbox_inches="tight")
    plt.show()


# Show waterfall for highest-scoring fraud case in sample
fraud_indices = np.where(
    xgb_app.predict_proba(X_app_test)[:, 1] > 0.7
)[0]

if len(fraud_indices) > 0:
    explain_single_prediction(
        xgb_app, explainer_app,
        X_app_test, APP_FEATURE_NAMES,
        "applications", example_idx=fraud_indices[0]
    )

fraud_indices_txn = np.where(
    xgb_txn.predict_proba(X_txn_test)[:, 1] > 0.7
)[0]

if len(fraud_indices_txn) > 0:
    explain_single_prediction(
        xgb_txn, explainer_txn,
        X_txn_test, TXN_FEATURE_NAMES,
        "transactions", example_idx=fraud_indices_txn[0]
    )